# 07 – Few-label supervised head fine-tuning
**Reproduces:** Section 5.7 (the "prediction-head fine-tuning" row of the few-label table) and Supplementary Section S8

**Responds to:** Reviewer 1 Comment 2 (additional exploratory few-label adaptation)

**What it does:** fine-tunes the prediction head of the trained source models (the 448 → 256 → 128 → 64 → 1 MLP) with a few labelled target records, using PyTorch rather than Ridge. The graph encoder, IMCA, pooling, ChemBERTa projector and cell-line embedding stay frozen (eval mode); the original ordered 448-dimensional head input is cached for both drug orders. Two settings are considered: last layer only (65 trainable parameters) and whole head (156,161). The setting and the number of epochs (0, 10, 30, 60, 100) are chosen only by 3-fold cross-validation inside the support set; epoch 0 (no update) is allowed. Controls: source-only prediction, a randomly initialised head on the same frozen source backbone, the target-only fingerprint Ridge, and the residual Ridge of notebook 06.

The design is that of notebook 06: three source seeds (42–44), five support draws (101–105), budgets 16/32/64/128 (primary: 64) and exactly the same support/query rows. This is exploratory, pair-disjoint target adaptation; it is neither zero-shot transfer nor a cold-drug evaluation after adaptation, and no setting is changed after looking at the query set.

**How to run:** `Runtime → Run all` (T4 recommended for the first feature extraction; CPU also works). The notebook is self-contained: Cell 2 writes the helper modules, so no ZIP is needed. It needs the project folder on Drive and the outputs of notebook 06 (`fewshot_extension_v1/`: frozen-feature `.npz` files, row-metadata CSVs, support/query manifests); the large `query_predictions.csv` files of notebook 06 are not needed. Results are written to `fewshot_head_finetune_v1/`. The run is resumable: every finished job is cached and reused when the notebook is run again with the same settings.

**Reading the output:** query metrics are not printed during the runs. Cell 5 prints the 64-label summary, the paired comparisons and the configurations selected by the support-set CV (epoch 0 = no update). The two Ridge controls of notebook 06 are recomputed on the same rows and checked against its `all_metrics.csv`. SDs are descriptive (support draws and source seeds are mixed); intervals are exploratory cluster bootstraps over query pairs and ignore shared-drug dependence.

In [ ]:
# ===== Cell 1: Environment and Drive paths =====
# Colab's own PyTorch is kept (not reinstalled).
!pip -q install rdkit torch-geometric scikit-learn scipy pandas matplotlib pillow
import os, sys, json, subprocess, platform
from pathlib import Path
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/DualGATSynergy')
OLD = ROOT/'fewshot_extension_v1'
OUTPUT = ROOT/'fewshot_head_finetune_v1'
CHECKPOINT_DIR = ROOT/'cross_dataset'
SEEDS = (42,43,44)
BUDGETS = (16,32,64,128)
DRAWS = (101,102,103,104,105)
assert ROOT.is_dir(), f'Project folder not found: {ROOT}'
assert OLD.is_dir(), f'Outputs of notebook 06 not found: {OLD}'
TOOLS = Path('/content/dualgat_head_tools_v1')
TOOLS.mkdir(exist_ok=True)
sys.path.insert(0,str(TOOLS))
os.environ['DGS_ROOT']=str(ROOT)
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
OUTPUT.mkdir(exist_ok=True)
print('ROOT:',ROOT)
print('Notebook 06 results:',OLD)
print('New outputs:',OUTPUT)


Mounted at /content/drive
ROOT: /content/drive/MyDrive/DualGATSynergy
Notebook 06 results: /content/drive/MyDrive/DualGATSynergy/fewshot_extension_v1
New outputs: /content/drive/MyDrive/DualGATSynergy/fewshot_head_finetune_v1


##Helper modules
The next cell writes four helper files (`ft_core.py`, `ft_source_runtime.py`, `head_finetune.py`, `test_head_finetune.py`) to the temporary Colab disk. It does not change any file of the previous experiment. The code is stored inside the cell, so no extra file has to be uploaded.

In [ ]:
#@title Write the helper modules (run once; no upload needed)
(TOOLS/'ft_core.py').write_text('"""Pair-disjoint target adaptation. No deep-learning dependencies.\nAll model selection is performed inside the labelled support budget.\nThis module never treats support resampling as independent test replication.\n"""\nimport hashlib, json\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom scipy.stats import pearsonr, spearmanr\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.linear_model import Ridge\nfrom sklearn.model_selection import KFold\nfrom sklearn.metrics import mean_squared_error, roc_auc_score, average_precision_score\n\nALPHAS = (0.1, 1., 10., 100., 1000.)\n\ndef lock_manifest(path, value):\n    path = Path(path)\n    if path.exists():\n        if json.loads(path.read_text()) != value:\n            raise ValueError(f\'Locked protocol differs: {path}. Use a new run directory, not an overwrite.\')\n    else:\n        path.parent.mkdir(parents=True, exist_ok=True)\n        path.write_text(json.dumps(value, indent=2, sort_keys=True))\n\n\ndef partition_pairs(meta, seed=2027, query_fraction=.5):\n    """Target pair split ignores labels. All cells/replicates of a pair stay together."""\n    groups = np.array(sorted(meta.pair_key.unique()))\n    if len(groups) < 20:\n        raise ValueError(\'Too few independent target pairs for the predeclared experiment.\')\n    rng = np.random.default_rng(seed)\n    groups = rng.permutation(groups)\n    nq = int(np.ceil(query_fraction * len(groups)))\n    query = meta.pair_key.isin(groups[:nq]).to_numpy()\n    assert not set(meta.loc[query, \'pair_key\']) & set(meta.loc[~query, \'pair_key\'])\n    return np.flatnonzero(~query), np.flatnonzero(query)\n\n\ndef draw_support(meta, pool, max_budget, seed):\n    """One labelled row per distinct unordered pair; prefixes give nested budgets.\n    Select pairs and rows without looking at response labels; not per-cell K-shot.\n    """\n    rng = np.random.default_rng(seed)\n    groups = rng.permutation(sorted(meta.iloc[pool].pair_key.unique()))\n    if len(groups) < max_budget:\n        raise ValueError(f\'Budget {max_budget} exceeds {len(groups)} support-pool pairs.\')\n    chosen=[]\n    for group in groups[:max_budget]:\n        candidates=pool[meta.iloc[pool].pair_key.to_numpy()==group]\n        chosen.append(int(rng.choice(candidates)))\n    return np.array(chosen)\n\n\ndef fit_ridge(x, y, alpha):\n    # Fit normalization on current training/support rows ONLY.\n    scaler=StandardScaler().fit(x)\n    model=Ridge(alpha=float(alpha), solver=\'cholesky\').fit(scaler.transform(x), y)\n    return scaler, model\n\n\ndef select_and_predict(xs, ys, xq, alphas=ALPHAS):\n    """Support-only inner CV; alpha chosen by MSE, never query PCC.\n    Support rows have unique pair keys, so KFold is also pair-disjoint.\n    """\n    xs=np.asarray(xs, dtype=float); ys=np.asarray(ys, dtype=float)\n    if len(ys)<6: raise ValueError(\'At least six labelled support pairs are required.\')\n    cv=list(KFold(n_splits=3, shuffle=True, random_state=818).split(xs))\n    losses=[]\n    for alpha in alphas:\n        loss=[]\n        for tr, va in cv:\n            scaler, model=fit_ridge(xs[tr], ys[tr], alpha)\n            loss.append(mean_squared_error(ys[va], model.predict(scaler.transform(xs[va]))))\n        losses.append(float(np.mean(loss)))\n    alpha=float(alphas[int(np.argmin(losses))])\n    scaler, model=fit_ridge(xs, ys, alpha)\n    return model.predict(scaler.transform(xq)), alpha\n\n\ndef score(y, p, raw, cells):\n    y=np.asarray(y); p=np.asarray(p); raw=np.asarray(raw); cells=np.asarray(cells)\n    corr=lambda a,b: float(pearsonr(a,b).statistic) if np.std(a)>1e-10 and np.std(b)>1e-10 else np.nan\n    binary=raw>10\n    out=dict(pcc=corr(y,p), scc=float(spearmanr(y,p).statistic) if np.std(p)>1e-10 else np.nan,\n             rmse_source_z=float(np.sqrt(mean_squared_error(y,p))), prevalence=float(binary.mean()))\n    out[\'auc\']=float(roc_auc_score(binary,p)) if np.unique(binary).size==2 else np.nan\n    out[\'ap\']=float(average_precision_score(binary,p)) if binary.any() else np.nan\n    within=[]\n    for cell in np.unique(cells):\n        m=cells==cell\n        if m.sum()>=30:\n            r=corr(y[m],p[m])\n            if np.isfinite(r):within.append((int(m.sum()),r))\n    out[\'within_cell_pcc\']=float(np.average([r for n,r in within],weights=[n for n,r in within])) if within else np.nan\n    out[\'within_cell_eligible_rows\']=sum(n for n,r in within)\n    return out\n\n\ndef run_bundle(bundle_path, out_dir, budgets=(16,32,64,128), draws=(101,102,103,104,105), split_seed=2027):\n    """Fit adapters, then evaluate fixed query. Bundle consists of .npz plus .csv.\n    y is standardized with SOURCE training statistics. No target-wide scaling.\n    """\n    bundle_path=Path(bundle_path); out=Path(out_dir);out.mkdir(parents=True,exist_ok=True)\n    z=np.load(bundle_path,allow_pickle=False)\n    meta=pd.read_csv(bundle_path.with_suffix(\'.csv\'), dtype={\'row_id\':str,\'pair_key\':str,\'cell_line\':str})\n    required={\'base\',\'uniform\',\'independent\',\'interaction\',\'fingerprint\',\'label\',\'raw_target\'}\n    assert required<=set(z.files)\n    assert all(len(z[k])==len(meta) for k in required)\n    assert meta.row_id.is_unique\n    for k in required:\n        if not np.isfinite(z[k]).all(): raise ValueError(\'Nonfinite bundle values: \'+k)\n    pool,q=partition_pairs(meta,seed=split_seed)\n    manifest=dict(protocol=\'pair_disjoint_one_labelled_row_per_support_pair_v1\', split_seed=int(split_seed),\n                  budgets=list(map(int,budgets)), draws=list(map(int,draws)), alphas=list(ALPHAS),\n                  query_rows=meta.iloc[q].row_id.tolist(),pool_rows=meta.iloc[pool].row_id.tolist(),\n                  feature_sha256=hashlib.sha256(bundle_path.read_bytes()).hexdigest())\n    lock_manifest(out/\'locked_protocol.json\',manifest)\n    # Query labels are not passed to any fitting or alpha-selection function.\n    y=z[\'label\']; base=z[\'base\']; full=np.concatenate([z[\'independent\'],z[\'interaction\']],axis=1)\n    rows=[]; records=[]\n    for ds in draws:\n        support_order=draw_support(meta,pool,max(budgets),ds)\n        for budget in budgets:\n            s=support_order[:budget]\n            assert len(set(meta.iloc[s].pair_key))==budget\n            assert not set(meta.iloc[s].pair_key)&set(meta.iloc[q].pair_key)\n            support_info=dict(draw=int(ds),budget=int(budget),\n                              n_support_rows=len(s),n_support_pairs=meta.iloc[s].pair_key.nunique(),\n                              n_support_cells=meta.iloc[s].cell_line.nunique(),\n                              n_query_rows=len(q),n_query_pairs=meta.iloc[q].pair_key.nunique())\n            support_info[\'support_rows\']=meta.iloc[s].row_id.tolist()\n            lock_manifest(out/f\'support_draw{ds}_n{budget}.json\',support_info)\n            predictions={\'source_only_sym\':(base[q],np.nan),\n                         \'source_uniform_attention_intervention\':(z[\'uniform\'][q],np.nan),\n                         \'support_mean_residual\':(base[q]+float(np.mean(y[s]-base[s])),np.nan)}\n            # This is an ADAPTER FEATURE ablation, NOT retraining/removing source IMCA.\n            for name,x,residual in [\n                (\'target_only_fingerprint_ridge\',z[\'fingerprint\'],False),\n                (\'target_head_frozen_full\',full,False),\n                (\'residual_independent_features\',z[\'independent\'],True),\n                (\'residual_full_features\',full,True)]:\n                fit_y=y[s]-base[s] if residual else y[s]\n                p,alpha=select_and_predict(x[s],fit_y,x[q])\n                predictions[name]=(p+base[q] if residual else p,alpha)\n            for name,(p,alpha) in predictions.items():\n                metrics=score(y[q],p,z[\'raw_target\'][q],meta.iloc[q].cell_line.to_numpy())\n                rows.append({**{k:v for k,v in support_info.items() if k!=\'support_rows\'},\'method\':name,\'alpha\':alpha,**metrics})\n                records.append(pd.DataFrame(dict(row_id=meta.iloc[q].row_id.to_numpy(),pair_key=meta.iloc[q].pair_key.to_numpy(),\n                    cell_line=meta.iloc[q].cell_line.to_numpy(),draw=ds,budget=budget,method=name,\n                    target=y[q],raw_target=z[\'raw_target\'][q],prediction=p)))\n    metrics=pd.DataFrame(rows);metrics.to_csv(out/\'metrics.csv\',index=False)\n    pd.concat(records,ignore_index=True).to_csv(out/\'query_predictions.csv.gz\',index=False)\n    # Descriptive only: these draws share the same query and are NOT independent experiments.\n    metrics.groupby([\'budget\',\'method\']).agg(pcc_mean=(\'pcc\',\'mean\'),pcc_support_draw_sd=(\'pcc\',\'std\'),\n                ap_mean=(\'ap\',\'mean\'),within_cell_mean=(\'within_cell_pcc\',\'mean\')).to_csv(out/\'descriptive_summary.csv\')\n    return metrics\n\n\ndef paired_pair_bootstrap(predictions, method_a=\'residual_full_features\',method_b=\'residual_independent_features\',\n                          budget=128,n_boot=2000,seed=919):\n    """Average paired PCC differences across training seeds/support draws within each resample.\n    One shared cluster resample across all runs. Pair clusters ignore shared-drug dependence.\n    predictions must contain seed, draw, budget, method, row_id, pair_key, target, prediction.\n    """\n    d=predictions[predictions.budget==budget]\n    ids=[\'seed\',\'draw\',\'row_id\',\'pair_key\',\'target\']\n    a=d[d.method==method_a][ids+[\'prediction\']].rename(columns={\'prediction\':\'a\'})\n    b=d[d.method==method_b][ids+[\'prediction\']].rename(columns={\'prediction\':\'b\'})\n    merged=a.merge(b,on=ids,validate=\'one_to_one\')\n    assert len(merged)==len(a)==len(b)\n    groups=np.array(sorted(merged.pair_key.unique()))\n    if len(groups)<20:raise ValueError(\'Too few pairs for a useful interval.\')\n    runs=[]\n    for _,r in merged.groupby([\'seed\',\'draw\']):\n        r=r.reset_index(drop=True)\n        runs.append((r, {g:np.flatnonzero(r.pair_key.to_numpy()==g) for g in groups}))\n    def stat(sample):\n        vals=[]\n        for r,lookup in runs:\n            ix=np.concatenate([lookup[g] for g in sample]); t=r.target.to_numpy()[ix]\n            if np.std(t)>1e-10 and np.std(r.a.to_numpy()[ix])>1e-10 and np.std(r.b.to_numpy()[ix])>1e-10:\n                vals.append(pearsonr(t,r.a.to_numpy()[ix]).statistic-pearsonr(t,r.b.to_numpy()[ix]).statistic)\n        return float(np.mean(vals)) if vals else np.nan\n    estimate=stat(groups);rng=np.random.default_rng(seed)\n    boots=np.array([stat(rng.choice(groups,len(groups),replace=True)) for _ in range(n_boot)])\n    lo,hi=np.nanquantile(boots,[.025,.975])\n    return dict(method_a=method_a,method_b=method_b,budget=budget,delta_pcc=estimate,\n                ci_low=float(lo),ci_high=float(hi),n_pairs=len(groups),\n                warning=\'Exploratory; fixed query, few source seeds; shared-drug dependence not modeled.\')\n', encoding="utf-8")
(TOOLS/'ft_source_runtime.py').write_text('# Reused verbatim from notebook 04 (selected data-preparation and model-definition cells only).\n# The cell numbers below are those of notebook 04. No source retraining, test-evaluation loop or manuscript file is run here.\n# ===== Cell 1: Environment =====\n# Colab only. The package versions used for the reported results are pinned in requirements.txt.\n\n# Standard library\nimport io, json, os, sys\nfrom pathlib import Path\n\n# Numerics and data\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nfrom scipy.stats import pearsonr, ttest_rel\n\n# Deep learning\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.optim.lr_scheduler import ReduceLROnPlateau\nfrom torch.utils.data import Dataset\n\n# Graph learning\nfrom torch_geometric.data import Data, Batch\nfrom torch_geometric.loader import DataLoader\nfrom torch_geometric.nn import GATv2Conv\nfrom torch_geometric.nn.aggr import AttentionalAggregation\nfrom torch_geometric.utils import to_dense_batch\n\n# Chemistry\nfrom rdkit import Chem, RDLogger, Geometry\nfrom rdkit.Chem import AllChem\nfrom rdkit.Chem.Draw import SimilarityMaps, rdMolDraw2D\nfrom PIL import Image\nfrom rdkit.Chem import Draw, rdDepictor\n\n\n# ChemBERTa (transformers) is imported only where it is needed, in Cell 5 below.\n\n# Classical ML and metrics\nfrom sklearn.ensemble import RandomForestRegressor\nfrom sklearn.metrics import average_precision_score, mean_squared_error, roc_auc_score\nfrom sklearn.preprocessing import StandardScaler\n\nRDLogger.DisableLog("rdApp.*")\nROOT = Path(os.environ[\'DGS_ROOT\'])\nDATA_DIR=ROOT/\'data\'\nDEVICE=\'cuda\' if torch.cuda.is_available() else \'cpu\'\nSEED=42\nXD_SETTINGS=(\'unified_loewe\',)\nXD_SEEDS=(42,43,44)\nXD_BASELINES=False\n\n\n# ===== Cell 3: Load and filter the raw data =====\n# --- 3.1 DrugComb v1.5 ---\nDRUGCOMB_COLS = ["drug_row", "drug_col", "cell_line_name", "synergy_hsa", "study_name", "tissue_name"]\ndrugcomb_raw = pd.read_csv(DATA_DIR / "drugcomb_full_v1.5.csv", usecols=DRUGCOMB_COLS, low_memory=False)\nprint(f"Raw records:                 {len(drugcomb_raw):,}   (paper: 1,432,351)")\n\n# Remove non-oncology studies\nNON_ONCOLOGY_KEYWORDS = ["malaria", "covid", "sars", "plasmodium", "tuberculosis"]\nis_non_oncology = (drugcomb_raw["study_name"].astype(str).str.lower()\n                   .str.contains("|".join(NON_ONCOLOGY_KEYWORDS), na=False))\ndrugcomb_raw = drugcomb_raw[~is_non_oncology].copy()\n\n# Drop records without an HSA score or a cell-line name (removes nothing in v1.5)\ndrugcomb_raw = drugcomb_raw.dropna(subset=["synergy_hsa", "cell_line_name"])\nn_compounds = len(set(drugcomb_raw["drug_row"]) | set(drugcomb_raw["drug_col"]))\nprint(f"After filtering:             {len(drugcomb_raw):,}   (paper: 1,432,146) | compounds: {n_compounds:,} (paper: 4,608)")\n\n# --- 3.2 Compound -> SMILES cache (resolved through PubChem) ---\ncache = pd.read_csv(DATA_DIR / "smiles_cache.csv")        # columns: name, smiles (empty if not resolved)\ndrug_to_smiles = dict(zip(cache["name"], cache["smiles"]))\nknown_drugs = {name for name, smi in drug_to_smiles.items() if isinstance(smi, str) and smi}\nprint(f"Cache: {len(drug_to_smiles)} names, {len(known_drugs)} resolved   (paper: 452 / 418)")\n\n# Keep only records where BOTH compounds have a SMILES string\ndrugcomb = drugcomb_raw[drugcomb_raw["drug_row"].isin(known_drugs) &\n                        drugcomb_raw["drug_col"].isin(known_drugs)].copy()\nn_drugs = len(set(drugcomb["drug_row"]) | set(drugcomb["drug_col"]))\nprint(f"DrugComb subset: {len(drugcomb):,} records, {n_drugs} drugs   (paper: 61,111 / 276)")\n\n# --- 3.3 O\'Neil ---\noneil      = pd.read_csv(DATA_DIR / "oneil_summary_idx.csv")\ndrugs_meta = pd.read_csv(DATA_DIR / "drugs.csv")\ncells_meta = pd.read_csv(DATA_DIR / "cell_lines.csv")\ndrug_map = dict(zip(drugs_meta["id"], drugs_meta["name"]))\ncell_map = dict(zip(cells_meta["id"], cells_meta["name"]))\noneil["drug_row"]       = oneil["drug_row_idx"].map(drug_map)\noneil["drug_col"]       = oneil["drug_col_idx"].map(drug_map)\noneil["cell_line_name"] = oneil["cell_line_idx"].map(cell_map)\nprint(f"O\'Neil: {len(oneil):,} records   (paper: 22,737) | unmapped names: "\n      f"{int(oneil[[\'drug_row\', \'drug_col\', \'cell_line_name\']].isna().sum().sum())}")\n\n# ===== Cell 4: O\'Neil records with SMILES =====\noneil["smiles_row"] = oneil["drug_row"].map(drug_to_smiles)\noneil["smiles_col"] = oneil["drug_col"].map(drug_to_smiles)\noneil_processed = oneil.dropna(subset=["smiles_row", "smiles_col"])[\n    ["drug_row", "drug_col", "smiles_row", "smiles_col", "cell_line_name", "synergy_loewe"]].copy()\noneil_processed.columns = ["drug_1", "drug_2", "smiles_1", "smiles_2", "cell_line", "synergy_score"]\nprint(f"O\'Neil: {len(oneil_processed):,} of {len(oneil):,} records have SMILES for both drugs   (paper: 22,737 of 22,737)")\n\n# ===== Cell 5: ChemBERTa embeddings (frozen pretrained model, [CLS] token) =====\nCHEMBERTA_NAME       = "seyonec/ChemBERTa-zinc-base-v1"\nCHEMBERTA_SHIPPED    = DATA_DIR / "drug_to_chemberta.pt"               # 418 compounds x 768 numbers (about 1.3 MB)\nCHEMBERTA_RECOMPUTED = DATA_DIR / "drug_to_chemberta_recomputed.pt"    # never overwrites the shipped file\nRECOMPUTE_CHEMBERTA  = False   # False: load the shipped file (no download, CPU is fine)\n                               # True : recompute with the pretrained model (inference only, no training)\n\nif not RECOMPUTE_CHEMBERTA:\n    drug_to_chemberta = torch.load(CHEMBERTA_SHIPPED, map_location="cpu")\nelse:\n    from transformers import AutoTokenizer, AutoModel\n    tokenizer = AutoTokenizer.from_pretrained(CHEMBERTA_NAME)\n    chemberta = AutoModel.from_pretrained(CHEMBERTA_NAME).eval().to(DEVICE)\n    print("Model revision:", getattr(chemberta.config, "_commit_hash", "unknown"))\n\n    @torch.no_grad()\n    def chemberta_embedding(smiles):\n        """[CLS]-token embedding of one SMILES string (truncated to 128 tokens)."""\n        inputs = tokenizer(smiles, return_tensors="pt", padding=True, truncation=True, max_length=128).to(DEVICE)\n        return chemberta(**inputs).last_hidden_state[:, 0, :].squeeze(0).cpu()\n\n    drug_to_chemberta = {name: chemberta_embedding(drug_to_smiles[name]) for name in sorted(known_drugs)}\n    torch.save(drug_to_chemberta, CHEMBERTA_RECOMPUTED)\n\nCHEMBERTA_DIM = next(iter(drug_to_chemberta.values())).shape[0]\nprint(f"ChemBERTa embeddings: {len(drug_to_chemberta)} compounds | dim = {CHEMBERTA_DIM}   (paper pipeline: 418 | 768)")\nassert set(drug_to_chemberta) == known_drugs, "embedding file and SMILES cache do not match"\n\n# ===== Cell 6: Molecular graphs =====\nNODE_FEATURES = ["atomic_number", "degree", "explicit_valence", "implicit_valence", "is_aromatic", "in_ring",\n                 "formal_charge", "num_radical_electrons", "hybridization", "chiral_tag", "atomic_mass"]\nNODE_DIM, EDGE_DIM = len(NODE_FEATURES), 1      # edge feature: bond order (1.0, 1.5, 2.0 or 3.0)\n\ndef _valences(atom):\n    """(explicit, implicit) valence. GetExplicitValence/GetImplicitValence produced the reported results;\n    newer RDKit versions replace them with GetValence(...)."""\n    try:\n        return atom.GetExplicitValence(), atom.GetImplicitValence()\n    except AttributeError:\n        return atom.GetValence(Chem.ValenceType.EXPLICIT), atom.GetValence(Chem.ValenceType.IMPLICIT)\n\ndef smiles_to_graph(smiles):\n    """SMILES string -> PyG graph (node features: NODE_FEATURES; edge feature: bond order).\n    Returns None if the SMILES is missing or cannot be parsed."""\n    if not isinstance(smiles, str) or not smiles:\n        return None\n    mol = Chem.MolFromSmiles(smiles)\n    if mol is None:\n        return None\n\n    features = []\n    for atom in mol.GetAtoms():\n        explicit_valence, implicit_valence = _valences(atom)\n        features.append([\n            atom.GetAtomicNum(),\n            atom.GetDegree(),\n            explicit_valence,\n            implicit_valence,\n            int(atom.GetIsAromatic()),\n            int(atom.IsInRing()),\n            atom.GetFormalCharge(),\n            atom.GetNumRadicalElectrons(),\n            int(atom.GetHybridization()),\n            int(atom.GetChiralTag()),       # RDKit chiral tag (none / clockwise / counter-clockwise / other)\n            atom.GetMass(),\n        ])\n    x = torch.tensor(features, dtype=torch.float)\n\n    edge_index, edge_attr = [], []\n    for bond in mol.GetBonds():\n        s, e = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()\n        order = bond.GetBondTypeAsDouble()\n        edge_index += [(s, e), (e, s)]           # undirected: both directions\n        edge_attr  += [order, order]\n\n    if edge_index:\n        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()\n        edge_attr  = torch.tensor(edge_attr,  dtype=torch.float).view(-1, 1)\n    else:                                         # single-atom molecule\n        edge_index = torch.empty((2, 0), dtype=torch.long)\n        edge_attr  = torch.empty((0, 1),   dtype=torch.float)\n    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)\n\n# Sanity check on all resolved compounds\ngraph_check = {name: smiles_to_graph(drug_to_smiles[name]) for name in known_drugs}\nn_failed   = sum(g is None for g in graph_check.values())\nn_no_bonds = sum(g is not None and g.edge_index.shape[1] == 0 for g in graph_check.values())\nprint(f"{len(graph_check)} cached compounds | failed to parse: {n_failed} | without any bond: {n_no_bonds} "\n      f"(the paper\'s statement concerns the 309 compounds used in the experiments; see the check after Cell 7)")\nassert all(g.x.shape[1] == NODE_DIM for g in graph_check.values() if g is not None)\ndel graph_check\n\n# ===== Cell 7: Combine DrugComb (HSA) and O\'Neil (Loewe) =====\ndrugcomb_unified = (drugcomb.dropna(subset=["synergy_hsa", "cell_line_name"])\n                    [["drug_row", "drug_col", "synergy_hsa", "cell_line_name"]]\n                    .rename(columns={"drug_row": "drug_1", "drug_col": "drug_2",\n                                     "synergy_hsa": "raw_target", "cell_line_name": "cell_line"}))\ndrugcomb_unified["dataset_type"] = "drugcomb"\n\noneil_unified = (oneil_processed.dropna(subset=["synergy_score"])\n                 [["drug_1", "drug_2", "synergy_score", "cell_line"]]\n                 .rename(columns={"synergy_score": "raw_target"}))\noneil_unified["dataset_type"] = "oneil"\n\ndf_combined = pd.concat([drugcomb_unified, oneil_unified], ignore_index=True)\n\n# One integer per cell line (order of first appearance) and the list of all compounds\nall_cell_lines   = df_combined["cell_line"].unique()\ncell_line_to_idx = {cl: i for i, cl in enumerate(all_cell_lines)}\nn_cell_lines     = len(all_cell_lines)\nall_drugs        = pd.concat([df_combined["drug_1"], df_combined["drug_2"]]).unique()\n\nprint(f"Records: {len(df_combined):,} (paper: 83,848) | cell-line identifiers: {n_cell_lines} (paper: 251) "\n      f"| compounds: {len(all_drugs)} (paper: 309)")\n\n# ===== Cell 9: Dataset =====\n_mol_graph_cache = {}\n\ndef get_cached_graph(smiles):\n    """Build each molecular graph once; return a copy so per-sample attributes never leak into the cache."""\n    if smiles not in _mol_graph_cache:\n        _mol_graph_cache[smiles] = smiles_to_graph(smiles)\n    cached = _mol_graph_cache[smiles]\n    return cached.clone() if cached is not None else None\n\n\nclass UnifiedSynergyDataset(Dataset):\n    """Sample = (graph of drug 1, graph of drug 2, standardized synergy label).\n    A record is kept only if both drugs have a ChemBERTa embedding and a SMILES that RDKit can parse."""\n    def __init__(self, dataframe, drug_smiles_dict, drug_chemberta_dict, cell_line_to_idx):\n        self.data        = dataframe.reset_index(drop=True)\n        self.drug_smiles = drug_smiles_dict\n        self.drug_cb     = drug_chemberta_dict\n        self.cell_to_idx = cell_line_to_idx\n\n        valid_drugs = {name for name in drug_chemberta_dict\n                       if isinstance(drug_smiles_dict.get(name), str)\n                       and Chem.MolFromSmiles(drug_smiles_dict[name]) is not None}\n        keep = self.data["drug_1"].isin(valid_drugs) & self.data["drug_2"].isin(valid_drugs)\n        self.valid_indices = list(self.data.index[keep])\n        print(f"Valid samples: {len(self.valid_indices):,} of {len(self.data):,}")\n\n    def __len__(self):\n        return len(self.valid_indices)\n\n    def __getitem__(self, idx):\n        row = self.data.iloc[self.valid_indices[idx]]\n        g1 = get_cached_graph(self.drug_smiles[row["drug_1"]])\n        g2 = get_cached_graph(self.drug_smiles[row["drug_2"]])\n        g1.cb = self.drug_cb[row["drug_1"]].unsqueeze(0)\n        g2.cb = self.drug_cb[row["drug_2"]].unsqueeze(0)\n        cl_idx = self.cell_to_idx[row["cell_line"]]\n        g1.cell_idx = torch.tensor([cl_idx], dtype=torch.long)\n        g2.cell_idx = torch.tensor([cl_idx], dtype=torch.long)\n        target = torch.tensor(row["label"], dtype=torch.float)\n        return g1, g2, target\n\n# ===== Cell 10: DualGATSynergy =====\nclass DualGATSynergy(nn.Module):\n    def __init__(self, num_atom_features, chemberta_dim, n_cell_lines, hidden_dim=64, heads=2):\n        super().__init__()\n        # Weight-shared two-layer GATv2 atom encoder (bond order is the edge feature)\n        self.conv1 = GATv2Conv(num_atom_features, hidden_dim, heads=heads, edge_dim=1, dropout=0.2)\n        self.conv2 = GATv2Conv(hidden_dim * heads, hidden_dim, heads=1, concat=False, edge_dim=1, dropout=0.2)\n\n        # Gated-attention pooling (one gate network, used for all four pooled streams)\n        self.pool = AttentionalAggregation(gate_nn=nn.Sequential(\n            nn.Linear(hidden_dim, 32), nn.ReLU(), nn.Linear(32, 1)))\n\n        # IMCA: 4-head atom-level cross-attention; the same weights are used in both directions\n        self.cross_attention = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=4, dropout=0.1, batch_first=True)\n\n        # Projection of the 768-d ChemBERTa vector; LayerNorm aligns its scale with the GAT vectors\n        self.chemberta_proj = nn.Sequential(\n            nn.Linear(chemberta_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.ReLU(), nn.Dropout(0.3))\n\n        self.cell_embedding = nn.Embedding(n_cell_lines, hidden_dim)\n\n        # 7 streams x 64 = 448 inputs\n        self.predictor = nn.Sequential(\n            nn.Linear(hidden_dim * 7, 256), nn.ReLU(), nn.Dropout(0.3),\n            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.3),\n            nn.Linear(128, 64), nn.ReLU(),\n            nn.Linear(64, 1))\n\n    def encode_atoms(self, x, edge_index, edge_attr):\n        x = F.elu(self.conv1(x, edge_index, edge_attr=edge_attr))\n        x = F.dropout(x, p=0.2, training=self.training)\n        return self.conv2(x, edge_index, edge_attr=edge_attr)\n\n    def forward(self, d1, d2):\n        atoms1 = self.encode_atoms(d1.x, d1.edge_index, d1.edge_attr)     # per-atom embeddings, before any pooling\n        atoms2 = self.encode_atoms(d2.x, d2.edge_index, d2.edge_attr)\n        emb1 = self.pool(atoms1, d1.batch)\n        emb2 = self.pool(atoms2, d2.batch)\n\n        dense1, mask1 = to_dense_batch(atoms1, d1.batch)                   # pad molecules to equal size\n        dense2, mask2 = to_dense_batch(atoms2, d2.batch)\n        attn_1to2, w_1to2 = self.cross_attention(dense1, dense2, dense2, key_padding_mask=~mask2)\n        attn_2to1, w_2to1 = self.cross_attention(dense2, dense1, dense1, key_padding_mask=~mask1)\n        pooled_1to2 = self.pool(attn_1to2[mask1], d1.batch)                # drop padded atoms, then pool\n        pooled_2to1 = self.pool(attn_2to1[mask2], d2.batch)\n\n        cb1_proj = self.chemberta_proj(d1.cb)\n        cb2_proj = self.chemberta_proj(d2.cb)\n        cell_emb = self.cell_embedding(d1.cell_idx)\n\n        combined = torch.cat([emb1, emb2, pooled_1to2, pooled_2to1, cb1_proj, cb2_proj, cell_emb], dim=1)\n        return self.predictor(combined), (w_1to2, w_2to1)\n\n\nMODEL_CFG = dict(num_atom_features=NODE_DIM, chemberta_dim=CHEMBERTA_DIM, n_cell_lines=n_cell_lines, hidden_dim=64)\n_m = DualGATSynergy(**MODEL_CFG)\nn_params = sum(p.numel() for p in _m.parameters() if p.requires_grad)\nprint(f"Trainable parameters: {n_params:,}   (paper: 260,482)")\nassert n_params == 260_482\ndel _m\n\n', encoding="utf-8")
(TOOLS/'head_finetune.py').write_text('"""Few-label head fine-tuning with support-only CV and exact inference order averaging.\nFrozen graph representations are cached in ORIGINAL ordered 448-dimensional form.\nNo graph weights, ChemBERTa projectors, pooling, attention or cell embeddings are updated.\n"""\nimport copy, hashlib, json, os\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom sklearn.model_selection import KFold\nfrom ft_core import lock_manifest, score, select_and_predict\n\nEPOCH_GRID=(0,10,30,60,100)\nCANDIDATES=({\'mode\':\'last_layer\',\'lr\':1e-3},{\'mode\':\'whole_head\',\'lr\':1e-4})\nANCHOR=0.01\n\ndef stable_seed(*args):\n    return int(hashlib.sha256(\'|\'.join(map(str,args)).encode()).hexdigest()[:8],16) % (2**31-1)\n\ndef seed_all(seed):\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)\n\ndef make_head(input_dim=448):\n    return nn.Sequential(nn.Linear(input_dim,256),nn.ReLU(),nn.Dropout(.3),\n            nn.Linear(256,128),nn.ReLU(),nn.Dropout(.3),nn.Linear(128,64),nn.ReLU(),nn.Linear(64,1))\n\n@torch.no_grad()\ndef predict(head,xa,xb,device=\'cpu\',batch_size=2048):\n    head.eval();outputs=[]\n    for start in range(0,len(xa),batch_size):\n        a=torch.as_tensor(xa[start:start+batch_size],dtype=torch.float32,device=device)\n        b=torch.as_tensor(xb[start:start+batch_size],dtype=torch.float32,device=device)\n        outputs.append(((head(a)+head(b))*.5).flatten().cpu().numpy())\n    return np.concatenate(outputs)\n\ndef fit_path(initial,xa,xb,y,config,epochs,seed,device=\'cpu\',validation=None,anchor=ANCHOR):\n    """Only support training labels are accepted. Validation is a support-inner fold.\n    Entire support is one mini-batch. Epoch 0 is a legitimate no-update candidate.\n    """\n    seed_all(seed);head=copy.deepcopy(initial).to(device)\n    for p in head.parameters():p.requires_grad_(False)\n    trainable=list(head[-1].parameters()) if config[\'mode\']==\'last_layer\' else list(head.parameters())\n    if config[\'mode\'] not in (\'last_layer\',\'whole_head\'):raise ValueError(\'Unknown mode\')\n    for p in trainable:p.requires_grad_(True)\n    anchors=[p.detach().clone() for p in trainable]\n    optimizer=torch.optim.Adam(trainable,lr=config[\'lr\'],weight_decay=0.)\n    a=torch.as_tensor(xa,dtype=torch.float32,device=device)\n    b=torch.as_tensor(xb,dtype=torch.float32,device=device)\n    t=torch.as_tensor(y,dtype=torch.float32,device=device).view(-1)\n    losses={}\n    for epoch in range(max(epochs)+1):\n        if epoch in epochs and validation is not None:\n            va,vb,vy=validation\n            p=predict(head,va,vb,device)\n            losses[epoch]=float(nn.functional.smooth_l1_loss(torch.as_tensor(p),torch.as_tensor(vy,dtype=torch.float32)))\n        if epoch==max(epochs):break\n        # Last-layer-only: keep all preceding frozen head layers deterministic, no dropout.\n        # Whole-head: retain original 0.3 dropout during updates, eval mode at prediction.\n        head.train(config[\'mode\']==\'whole_head\')\n        optimizer.zero_grad(set_to_none=True)\n        pred=((head(a)+head(b))*.5).flatten()\n        loss=nn.functional.smooth_l1_loss(pred,t)\n        if anchor:\n            loss=loss+anchor*sum((p-p0).square().sum() for p,p0 in zip(trainable,anchors))\n        if not torch.isfinite(loss):raise FloatingPointError(\'Nonfinite support loss\')\n        loss.backward();optimizer.step()\n    head.eval()\n    return head,losses\n\ndef select_on_support(initial,xa,xb,y,run_seed,device=\'cpu\'):\n    """Three inner folds, unique support pairs. No query features/labels accepted."""\n    if len(y)<6:raise ValueError(\'At least six unique labelled support pairs required.\')\n    folds=list(KFold(3,shuffle=True,random_state=818).split(xa));rows=[]\n    for ci,config in enumerate(CANDIDATES):\n        losses={epoch:[] for epoch in EPOCH_GRID}\n        for fi,(tr,va) in enumerate(folds):\n            _,path=fit_path(initial,xa[tr],xb[tr],y[tr],config,EPOCH_GRID,\n                stable_seed(run_seed,ci,fi),device,validation=(xa[va],xb[va],y[va]))\n            for epoch,value in path.items():losses[epoch].append(value)\n        for epoch,values in losses.items():\n            rows.append(dict(candidate=ci,mode=config[\'mode\'],lr=config[\'lr\'],epoch=epoch,\n                             support_cv_huber=float(np.mean(values))))\n    # Ties prefer fewer update steps, then last-layer-only. Query never used.\n    best=min(rows,key=lambda r:(r[\'support_cv_huber\'],r[\'epoch\'],r[\'candidate\']))\n    return best,rows\n\ndef adapt(initial,xa,xb,y,run_seed,device=\'cpu\'):\n    best,trace=select_on_support(initial,xa,xb,y,run_seed,device)\n    config={\'mode\':best[\'mode\'],\'lr\':best[\'lr\']}\n    trained,_=fit_path(initial,xa,xb,y,config,(best[\'epoch\'],),stable_seed(run_seed,\'refit\'),device)\n    return trained,best,trace\n\ndef export_caches(root,out,checkpoint_dir=None,seeds=(42,43,44),batch_size=128):\n    root=Path(root);out=Path(out);old=root/\'fewshot_extension_v1\'\n    ckptdir=Path(checkpoint_dir) if checkpoint_dir else root/\'cross_dataset\'\n    feature_dir=out/\'frozen_ordered_features\';feature_dir.mkdir(parents=True,exist_ok=True)\n    jobs=[]\n    for direction in (\'DrugComb_to_ONeil\',\'ONeil_to_DrugComb\'):\n        for seed in seeds:\n            stem=f\'{direction}_seed{seed}\'\n            paths=dict(old_npz=old/f\'{stem}.npz\',meta=old/f\'{stem}.csv\',\n                ckpt=ckptdir/f\'best_model_xdataset_unified_loewe_{stem}.pth\',\n                provenance=old/f\'{stem}_checkpoint.json\',protocol=old/f\'{stem}_adaptation\'/\'locked_protocol.json\')\n            for k,p in paths.items():\n                if not p.is_file():raise FileNotFoundError(f\'Missing {k}: {p}. Do not regenerate a different split.\')\n            checkpoint_sha=hashlib.sha256(paths[\'ckpt\'].read_bytes()).hexdigest()\n            if json.loads(paths[\'provenance\'].read_text())[\'sha256\']!=checkpoint_sha:\n                raise ValueError(\'Source checkpoint changed relative to experiment 06: \'+stem)\n            metadata=pd.read_csv(paths[\'meta\'],dtype={\'row_id\':str,\'pair_key\':str,\'cell_line\':str})\n            oldz=np.load(paths[\'old_npz\'],allow_pickle=False)\n            fingerprint=hashlib.sha256(paths[\'meta\'].read_bytes()).hexdigest()\n            cache=feature_dir/f\'{stem}.npz\'\n            if cache.exists():\n                c=np.load(cache,allow_pickle=False)\n                assert str(c[\'checkpoint_sha\'])==checkpoint_sha and str(c[\'metadata_sha\'])==fingerprint\n                assert np.array_equal(c[\'row_id\'],metadata.row_id.to_numpy(dtype=str))\n                assert np.allclose(c[\'base\'],oldz[\'base\'],atol=1e-3,rtol=1e-3)\n            jobs.append((direction,seed,paths,metadata,checkpoint_sha,fingerprint,cache))\n    if all(job[-1].exists() for job in jobs):\n        print(\'All frozen ordered feature caches exist; original raw CSV need not be read again.\')\n        return [str(job[-1]) for job in jobs]\n    os.environ[\'DGS_ROOT\']=str(root)\n    import ft_source_runtime as r\n    if Path(r.ROOT).resolve()!=root.resolve():raise RuntimeError(\'ROOT changed; restart runtime.\')\n    mapfile=root/\'splits\'/\'cell_line_to_idx.json\'\n    if mapfile.exists():\n        assert json.loads(mapfile.read_text())=={str(k):int(v) for k,v in r.cell_line_to_idx.items()},\'Cell index mismatch\'\n    from torch_geometric.loader import DataLoader\n    with torch.no_grad():\n        for direction,seed,paths,meta,checkpoint_sha,fingerprint,cache in jobs:\n            if cache.exists():continue\n            z=np.load(paths[\'old_npz\'],allow_pickle=False)\n            target=meta.copy();target[\'label\']=z[\'label\'];assert len(target)==len(z[\'base\'])\n            ds=r.UnifiedSynergyDataset(target,r.drug_to_smiles,r.drug_to_chemberta,r.cell_line_to_idx)\n            assert len(ds.valid_indices)==len(target),\'Molecular row validity changed\'\n            model=r.DualGATSynergy(**r.MODEL_CFG).to(r.DEVICE)\n            model.load_state_dict(torch.load(paths[\'ckpt\'],map_location=r.DEVICE,weights_only=True));model.eval()\n            xa=[];xb=[];pred=[]\n            captured=[]\n            def hook(module,args):captured.append(args[0].detach().cpu().numpy())\n            handle=model.predictor.register_forward_pre_hook(hook)\n            for a,b,y in DataLoader(ds,batch_size=batch_size,shuffle=False):\n                captured.clear();a,b=a.to(r.DEVICE),b.to(r.DEVICE)\n                pa,_=model(a,b);pb,_=model(b,a)\n                assert len(captured)==2\n                xa.append(captured[0]);xb.append(captured[1]);pred.append(((pa+pb)*.5).flatten().cpu().numpy())\n            handle.remove()\n            xa=np.concatenate(xa);xb=np.concatenate(xb);pred=np.concatenate(pred)\n            err=float(np.max(np.abs(pred-z[\'base\'])))\n            if err>1e-3:raise AssertionError(f\'{direction} seed {seed}: old source predictions differ by {err}; stop and inspect.\')\n            # Fresh standalone head must reproduce the source model on the cached inputs.\n            head=make_head().to(r.DEVICE)\n            state={k.removeprefix(\'predictor.\'):v for k,v in model.state_dict().items() if k.startswith(\'predictor.\')}\n            head.load_state_dict(state)\n            assert np.allclose(predict(head,xa,xb,r.DEVICE),pred,atol=2e-5,rtol=2e-4)\n            np.savez_compressed(cache,xa=xa,xb=xb,base=z[\'base\'],row_id=meta.row_id.to_numpy(dtype=str),\n                                checkpoint_sha=checkpoint_sha,metadata_sha=fingerprint)\n            print(\'CACHED\',direction,seed,\'rows\',len(meta),\'max source prediction discrepancy\',err)\n            del model,head\n            if r.DEVICE==\'cuda\':torch.cuda.empty_cache()\n    return [str(job[-1]) for job in jobs]\n\ndef weighted_pair_bootstrap(predictions,y,pair_keys,methods,primary=\'fewshot_pretrained_head\',n_boot=2000,seed=919):\n    """predictions [runs, methods, query rows]. Average correlations, never ensemble predictions.\n    Shared pair-cluster bootstrap implemented with sufficient statistics; no giant CSV needed.\n    """\n    p=np.asarray(predictions,dtype=float);y=np.asarray(y,dtype=float)\n    assert p.ndim==3 and p.shape[1]==len(methods) and p.shape[2]==len(y)\n    groups,idx=np.unique(pair_keys,return_inverse=True);g=len(groups)\n    if g<20:raise ValueError(\'Too few query pairs\')\n    count=np.bincount(idx,minlength=g).astype(float)\n    sy=np.bincount(idx,weights=y,minlength=g);sy2=np.bincount(idx,weights=y*y,minlength=g)\n    sp=np.empty((p.shape[0],p.shape[1],g));sp2=sp.copy();syp=sp.copy()\n    for r in range(p.shape[0]):\n        for m in range(p.shape[1]):\n            sp[r,m]=np.bincount(idx,weights=p[r,m],minlength=g)\n            sp2[r,m]=np.bincount(idx,weights=p[r,m]**2,minlength=g)\n            syp[r,m]=np.bincount(idx,weights=y*p[r,m],minlength=g)\n    def statistic(weights):\n        n=weights@count;ey=weights@sy;ey2=weights@sy2\n        ep=np.einsum(\'bg,rmg->brm\',weights,sp)\n        ep2=np.einsum(\'bg,rmg->brm\',weights,sp2)\n        eyp=np.einsum(\'bg,rmg->brm\',weights,syp)\n        cov=eyp-ey[:,None,None]*ep/n[:,None,None]\n        denom=np.sqrt(np.maximum(ey2-ey*ey/n,0)[:,None,None]*np.maximum(ep2-ep*ep/n[:,None,None],0))\n        corr=np.divide(cov,denom,out=np.full_like(cov,np.nan),where=denom>1e-12)\n        return np.nanmean(corr,axis=1)\n    est=statistic(np.ones((1,g)))[0]\n    rng=np.random.default_rng(seed)\n    samples=rng.multinomial(g,np.full(g,1/g),size=n_boot)\n    boot=statistic(samples);a=methods.index(primary);rows=[]\n    for b,name in enumerate(methods):\n        if b==a:continue\n        delta=boot[:,a]-boot[:,b];lo,hi=np.nanquantile(delta,[.025,.975])\n        rows.append(dict(method_a=primary,method_b=name,delta_pcc=float(est[a]-est[b]),\n            ci_low=float(lo),ci_high=float(hi),query_pairs=g,\n            warning=\'Exploratory; shared query and few source seeds; pair clusters ignore shared-drug dependence.\'))\n    return rows\n\ndef run_experiment(root,out,seeds=(42,43,44),budgets=(16,32,64,128),draws=(101,102,103,104,105),checkpoint_dir=None):\n    root=Path(root);out=Path(out);out.mkdir(parents=True,exist_ok=True);old=root/\'fewshot_extension_v1\'\n    ckptdir=Path(checkpoint_dir) if checkpoint_dir else root/\'cross_dataset\'\n    device=\'cuda\' if torch.cuda.is_available() else \'cpu\'\n    plan=dict(version=1,scope=\'Exploratory head-only target adaptation; same 06 pair-disjoint support/query; not cold-drug after adaptation\',\n       code_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),\n       primary_budget=64,budgets=list(budgets),draws=list(draws),source_seeds=list(seeds),\n       candidates=list(CANDIDATES),epoch_grid=list(EPOCH_GRID),anchor_l2_sum=ANCHOR,\n       cv=\'3-fold support-only, mean Huber loss; epoch zero allowed; no query selection\',\n       features=\'Frozen ORIGINAL ordered streams; symmetrized prediction used in both training and inference\',\n       random_head_control=\'Random head, same SOURCE-pretrained frozen backbone; not a target-only model\')\n    lock_manifest(out/\'analysis_plan.json\',plan)\n    allmetrics=[];allselections=[];alltraces=[];comparisons=[];audits=[]\n    methods=[\'source_only_sym\',\'fewshot_pretrained_head\',\'fewshot_random_head\',\'target_only_fingerprint_ridge\',\'residual_full_features_ridge\']\n    for direction in (\'DrugComb_to_ONeil\',\'ONeil_to_DrugComb\'):\n        primary_runs=[];primary_y=None;primary_pairs=None\n        for source_seed in seeds:\n            stem=f\'{direction}_seed{source_seed}\';meta=pd.read_csv(old/f\'{stem}.csv\',dtype={\'row_id\':str,\'pair_key\':str,\'cell_line\':str})\n            z=np.load(old/f\'{stem}.npz\',allow_pickle=False)\n            c=np.load(out/\'frozen_ordered_features\'/f\'{stem}.npz\',allow_pickle=False)\n            assert np.array_equal(c[\'row_id\'],meta.row_id.to_numpy(dtype=str))\n            protocol=json.loads((old/f\'{stem}_adaptation\'/\'locked_protocol.json\').read_text())\n            if not set(budgets)<=set(protocol[\'budgets\']) or not set(draws)<=set(protocol[\'draws\']):raise ValueError(\'Budget/draw not in original protocol\')\n            rowmap={v:i for i,v in enumerate(meta.row_id)}\n            q=np.array([rowmap[str(v)] for v in protocol[\'query_rows\']]);assert len(set(q))==len(q)\n            y=z[\'label\'];base=z[\'base\'];xa=c[\'xa\'];xb=c[\'xb\']\n            fp_features=z[\'fingerprint\']\n            full_features=np.concatenate([z[\'independent\'],z[\'interaction\']],axis=1)\n            assert np.allclose(c[\'base\'],base,atol=1e-3,rtol=1e-3)\n            checkpoint=ckptdir/f\'best_model_xdataset_unified_loewe_{stem}.pth\'\n            assert hashlib.sha256(checkpoint.read_bytes()).hexdigest()==str(c[\'checkpoint_sha\'])\n            state=torch.load(checkpoint,map_location=\'cpu\',weights_only=True)\n            pretrained=make_head();pretrained.load_state_dict({k.removeprefix(\'predictor.\'):v for k,v in state.items() if k.startswith(\'predictor.\')})\n            assert np.allclose(predict(pretrained,xa[q],xb[q]),base[q],atol=1e-3,rtol=1e-3)\n            if primary_y is None:primary_y=y[q];primary_pairs=meta.iloc[q].pair_key.to_numpy();primary_ids=meta.iloc[q].row_id.to_numpy()\n            assert np.array_equal(primary_ids,meta.iloc[q].row_id.to_numpy()) and np.allclose(primary_y,y[q])\n            for budget in budgets:\n                block=[]\n                for draw in draws:\n                    info=json.loads((old/f\'{stem}_adaptation\'/f\'support_draw{draw}_n{budget}.json\').read_text())\n                    s=np.array([rowmap[str(v)] for v in info[\'support_rows\']])\n                    assert len(s)==budget and len(set(meta.iloc[s].pair_key))==budget\n                    assert not set(meta.iloc[s].pair_key)&set(meta.iloc[q].pair_key)\n                    context=dict(direction=direction,seed=source_seed,budget=budget,draw=draw,\n                        n_support_rows=len(s),n_support_cells=meta.iloc[s].cell_line.nunique(),\n                        n_query_rows=len(q),n_query_pairs=meta.iloc[q].pair_key.nunique())\n                    run_seed=stable_seed(direction,source_seed,budget,draw)\n                    seed_all(stable_seed(run_seed,\'random_head\'))\n                    random_head=make_head()\n                    jobdir=out/\'job_cache\';jobdir.mkdir(exist_ok=True)\n                    jobfile=jobdir/f\'{stem}_n{budget}_draw{draw}.npz\'\n                    jobjson=jobfile.with_suffix(\'.json\')\n                    resumed=jobfile.exists() and jobjson.exists()\n                    if resumed:\n                        saved=np.load(jobfile,allow_pickle=False)\n                        info_saved=json.loads(jobjson.read_text())\n                        assert np.array_equal(saved[\'query_rows\'],meta.iloc[q].row_id.to_numpy(dtype=str))\n                        assert np.array_equal(saved[\'support_rows\'],meta.iloc[s].row_id.to_numpy(dtype=str))\n                        assert info_saved[\'checkpoint_sha\']==str(c[\'checkpoint_sha\'])\n                        predictions={name:saved[name] for name in methods}\n                        assert np.allclose(predictions[\'source_only_sym\'],base[q],atol=1e-8)\n                        allselections.extend(info_saved[\'selections\'])\n                        alltraces.extend(info_saved[\'traces\'])\n                    else:\n                        predictions={\'source_only_sym\':base[q]}\n                    job_selections=[];job_traces=[]\n                    for name,initial in ([] if resumed else [(\'fewshot_pretrained_head\',pretrained),(\'fewshot_random_head\',random_head)]):\n                        trained,best,trace=adapt(initial,xa[s],xb[s],y[s],stable_seed(run_seed,name),device)\n                        predictions[name]=predict(trained,xa[q],xb[q],device)\n                        selection={**context,\'method\':name,**best}\n                        traces=[{**context,\'method\':name,**row} for row in trace]\n                        allselections.append(selection);alltraces.extend(traces)\n                        job_selections.append(selection);job_traces.extend(traces)\n                        # Save updated HEAD only, never overwrite source checkpoint.\n                        folder=out/\'adapted_heads\';folder.mkdir(exist_ok=True)\n                        torch.save({k:v.detach().cpu() for k,v in trained.state_dict().items()},folder/f\'{stem}_n{budget}_draw{draw}_{name}.pth\')\n                        del trained\n                    # Recompute old controls with the identical support IDs and old CV rules.\n                    if not resumed:\n                        fp,_=select_and_predict(fp_features[s],y[s],fp_features[q])\n                        residual,_=select_and_predict(full_features[s],y[s]-base[s],full_features[q])\n                        predictions[\'target_only_fingerprint_ridge\']=fp\n                        predictions[\'residual_full_features_ridge\']=base[q]+residual\n                        np.savez_compressed(jobfile,**predictions,\n                            query_rows=meta.iloc[q].row_id.to_numpy(dtype=str),\n                            support_rows=meta.iloc[s].row_id.to_numpy(dtype=str))\n                        jobjson.write_text(json.dumps(dict(checkpoint_sha=str(c[\'checkpoint_sha\']),\n                            selections=job_selections,traces=job_traces),indent=2))\n                    for name,p in predictions.items():\n                        allmetrics.append({**context,\'method\':name,**score(y[q],p,z[\'raw_target\'][q],meta.iloc[q].cell_line.to_numpy())})\n                    block.append(np.stack([predictions[name] for name in methods]))\n                    print(direction,\'source seed\',source_seed,\'budget\',budget,\'support draw\',draw,\'done (no query metrics printed)\')\n                block=np.stack(block)\n                # Compact numeric backups, not repetitive multi-million-row CSVs.\n                pred_dir=out/\'compact_predictions\';pred_dir.mkdir(exist_ok=True)\n                np.savez_compressed(pred_dir/f\'{stem}_n{budget}.npz\',predictions=block.astype(np.float32),\n                    methods=np.asarray(methods),draws=np.asarray(draws),query_rows=meta.iloc[q].row_id.to_numpy(dtype=str),\n                    pair_keys=meta.iloc[q].pair_key.to_numpy(dtype=str),target=y[q],raw_target=z[\'raw_target\'][q])\n                if budget==64:primary_runs.extend(list(block))\n            audits.append(dict(direction=direction,seed=source_seed,checkpoint_sha=str(c[\'checkpoint_sha\']),\n                original_protocol_sha=hashlib.sha256((old/f\'{stem}_adaptation\'/\'locked_protocol.json\').read_bytes()).hexdigest(),\n                query_rows=len(q),query_pairs=meta.iloc[q].pair_key.nunique()))\n        if 64 in budgets:\n            rows=weighted_pair_bootstrap(np.stack(primary_runs),primary_y,primary_pairs,methods)\n            comparisons.extend(dict(direction=direction,budget=64,**r) for r in rows)\n    metrics=pd.DataFrame(allmetrics);metrics.to_csv(out/\'all_metrics.csv\',index=False)\n    oldmetrics=old/\'all_metrics.csv\'\n    if oldmetrics.exists():\n        previous=pd.read_csv(oldmetrics)\n        current=metrics[metrics.method.isin([\'source_only_sym\',\'target_only_fingerprint_ridge\',\'residual_full_features_ridge\'])].copy()\n        current[\'method\']=current.method.replace({\'residual_full_features_ridge\':\'residual_full_features\'})\n        keys=[\'direction\',\'seed\',\'budget\',\'draw\',\'method\']\n        check=current.merge(previous,on=keys,suffixes=(\'_new\',\'_old\'),validate=\'one_to_one\')\n        if len(check)!=len(current):raise AssertionError(\'Old baseline run coverage differs\')\n        check[\'pcc_abs_difference\']=abs(check.pcc_new-check.pcc_old)\n        check[keys+[\'pcc_new\',\'pcc_old\',\'pcc_abs_difference\']].to_csv(out/\'previous_control_reproduction_audit.csv\',index=False)\n        if check.pcc_abs_difference.max()>1e-4:\n            raise AssertionError(\'Old control PCC differs by >1e-4. Inspect reproduction audit before comparing new models.\')\n    selections=pd.DataFrame(allselections);selections.to_csv(out/\'support_only_selections.csv\',index=False)\n    pd.DataFrame(alltraces).to_csv(out/\'support_only_cv_trace.csv\',index=False)\n    pd.DataFrame(comparisons).to_csv(out/\'paired_primary_budget_comparisons.csv\',index=False)\n    pd.DataFrame(audits).to_csv(out/\'provenance_audit.csv\',index=False)\n    # SD here mixes support draws and training/source initialization; no independent-run significance claim.\n    summary=metrics.groupby([\'direction\',\'budget\',\'method\']).agg(pcc_mean=(\'pcc\',\'mean\'),pcc_run_sd=(\'pcc\',\'std\'),\n        scc_mean=(\'scc\',\'mean\'),within_cell_pcc=(\'within_cell_pcc\',\'mean\'),ap_mean=(\'ap\',\'mean\'),auc_mean=(\'auc\',\'mean\'))\n    summary.to_csv(out/\'all_descriptive_summary.csv\')\n    # Separate source-seed means and within-source support-draw variation.\n    metrics.groupby([\'direction\',\'budget\',\'method\',\'seed\']).agg(pcc_mean=(\'pcc\',\'mean\'),support_draw_sd=(\'pcc\',\'std\')).to_csv(out/\'per_source_seed_summary.csv\')\n    return summary\n', encoding="utf-8")
(TOOLS/'test_head_finetune.py').write_text('"""Synthetic SOFTWARE tests. Not drug-synergy performance results."""\nimport sys,copy,tempfile\nfrom pathlib import Path\nsys.path.insert(0,str(Path(__file__).resolve().parent))\nimport numpy as np\nimport torch\nfrom scipy.stats import pearsonr\nimport head_finetune as h\n\ndef main():\n    torch.set_num_threads(1);h.seed_all(31)\n    rng=np.random.default_rng(31);a=rng.normal(size=(24,448)).astype(\'float32\');b=rng.normal(size=(24,448)).astype(\'float32\')\n    original=h.make_head();original.eval();before={k:v.clone() for k,v in original.state_dict().items()}\n    baseline=h.predict(original,a,b);y=baseline+1.0\n    model,_=h.fit_path(original,a,b,y,dict(mode=\'last_layer\',lr=.02),(30,),seed=32,anchor=0.)\n    assert all(torch.equal(original.state_dict()[k],v) for k,v in before.items()),\'Source head was modified\'\n    assert torch.equal(model[0].weight,original[0].weight),\'Frozen first head layer changed\'\n    assert not torch.equal(model[-1].weight,original[-1].weight),\'Final layer failed to update\'\n    assert np.mean((h.predict(model,a,b)-y)**2)<np.mean((baseline-y)**2),\'Training did not learn toy offset\'\n    assert np.allclose(h.predict(model,a,b),h.predict(model,b,a),atol=1e-7),\'Inference not order invariant\'\n    unchanged,_=h.fit_path(original,a,b,y,dict(mode=\'last_layer\',lr=.02),(0,),seed=32)\n    assert np.allclose(h.predict(unchanged,a,b),baseline),\'Epoch 0 is not source model\'\n    whole,_=h.fit_path(original,a,b,y,dict(mode=\'whole_head\',lr=1e-3),(2,),seed=32)\n    assert not torch.equal(whole[0].weight,original[0].weight),\'Whole head did not update hidden layer\'\n    # Shortened CV grid for the smoke test only; delivered default stays unchanged.\n    saved_grid=h.EPOCH_GRID;h.EPOCH_GRID=(0,2,4)\n    model1,best1,tr1=h.adapt(original,a,b,y,run_seed=33)\n    model2,best2,tr2=h.adapt(original,a,b,y,run_seed=33)\n    h.EPOCH_GRID=saved_grid\n    assert best1==best2 and tr1==tr2\n    assert np.allclose(h.predict(model1,a,b),h.predict(model2,a,b))\n    assert best1[\'epoch\'] in (0,2,4)\n    # No query feature/label arguments exist in either fitting or CV selection API.\n    import inspect\n    assert \'query\' not in str(inspect.signature(h.adapt))\n    assert \'query\' not in str(inspect.signature(h.select_on_support))\n    n=120;target=rng.normal(size=n);pairs=np.repeat([f\'p{i}\' for i in range(30)],4)\n    pred=np.stack([np.stack([target+rng.normal(size=n)*.3,target+rng.normal(size=n)*.8]) for _ in range(3)])\n    rows=h.weighted_pair_bootstrap(pred,target,pairs,[\'fewshot_pretrained_head\',\'control\'],n_boot=40)\n    expected=np.mean([pearsonr(target,pred[r,0]).statistic-pearsonr(target,pred[r,1]).statistic for r in range(3)])\n    assert np.isclose(rows[0][\'delta_pcc\'],expected,atol=1e-10),\'Bootstrap point estimate differs from direct Pearson\'\n    assert rows[0][\'ci_low\']<=rows[0][\'ci_high\']\n    print(\'PASS: real PyTorch optimizer updates; last-layer freezing; whole-head updates; source-state immutability; exact eval order symmetry; epoch-zero baseline; deterministic support-only CV/refit; paired bootstrap vs direct Pearson.\')\n    print(\'Trainable parameter counts: final layer\',sum(p.numel() for p in original[-1].parameters()),\'; entire head\',sum(p.numel() for p in original.parameters()))\n\nif __name__==\'__main__\':main()\n', encoding="utf-8")
print("Helper modules written successfully.")

Helper modules written successfully.


##Software tests and environment record
The tests use synthetic data. A PASS only shows that the code works as designed (for example, frozen layers stay unchanged and epoch 0 gives the source prediction). It says nothing about synergy prediction performance. The real results come from Section 5.

In [ ]:
# ===== Cell 3: Software tests and environment record =====
# Synthetic-data tests of the helper code; they do not measure synergy prediction performance.
import runpy, torch
runpy.run_path(str(TOOLS/'test_head_finetune.py'),run_name='__main__')
print('PyTorch:',torch.__version__)
print('GPU available:',torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:',torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32=False
else:
    print('CPU execution is supported; a T4 GPU is recommended for the first graph-feature extraction.')
_ = (OUTPUT/'environment.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))

PASS: real PyTorch optimizer updates; last-layer freezing; whole-head updates; source-state immutability; exact eval order symmetry; epoch-zero baseline; deterministic support-only CV/refit; paired bootstrap vs direct Pearson.
Trainable parameter counts: final layer 65 ; entire head 156161
PyTorch: 2.11.0+cu130
GPU available: True
GPU: Tesla T4


##Extracting the original head inputs from the source models
This step only runs inference; the graph encoder and all other frozen parts are not changed. The checkpoint hashes, row IDs, cell-line index map and source predictions are checked against notebook 06. If the full cache already exists, it is reused.

A path or mismatch error stops the run on purpose. Do not replace the checkpoint or create a new split.

In [ ]:
# ===== Cell 4: Extract the original head inputs from the source models =====
# Inference only; an existing cache in frozen_ordered_features/ is checked and reused.
from head_finetune import export_caches
FEATURE_CACHES = export_caches(ROOT,OUTPUT,checkpoint_dir=CHECKPOINT_DIR,seeds=SEEDS,batch_size=128)
print('Ready frozen feature caches:',len(FEATURE_CACHES))

All frozen ordered feature caches exist; original raw CSV need not be read again.
Ready frozen feature caches: 6


##Fine-tuning and controls
The head setting and the number of epochs are selected only by 3-fold cross-validation inside the support set; the query set takes no part in this selection. The backbone is not trained in this step either. Predictions use the average over both drug orders in training and in evaluation, and in evaluation mode the result is exactly order-invariant.

The random-head control still uses the pretrained source backbone; the target-only control is the fingerprint Ridge. The two Ridge controls of notebook 06 are recomputed on the same row IDs and checked against the stored values. Each finished job is cached, so an interrupted run can be resumed with the same settings.

In [ ]:
# ===== Cell 5: Fine-tuning, controls and 64-label summary =====
# Resumable: every finished job is cached in job_cache/. Query metrics are not printed during the runs.
from head_finetune import run_experiment
SUMMARY = run_experiment(ROOT,OUTPUT,seeds=SEEDS,budgets=BUDGETS,draws=DRAWS,checkpoint_dir=CHECKPOINT_DIR)
print('Completed. Primary budget = 64 labelled target rows.')
display(SUMMARY.reset_index().query('budget == 64'))
import pandas as pd
display(pd.read_csv(OUTPUT/'paired_primary_budget_comparisons.csv'))
selection=pd.read_csv(OUTPUT/'support_only_selections.csv')
print('Support-CV selected configurations; epoch=0 means no neural update:')
display(selection.groupby(['method','mode','epoch']).size().rename('number_of_runs').reset_index())

DrugComb_to_ONeil source seed 42 budget 16 support draw 101 done (no query metrics printed)
DrugComb_to_ONeil source seed 42 budget 16 support draw 102 done (no query metrics printed)
DrugComb_to_ONeil source seed 42 budget 16 support draw 103 done (no query metrics printed)
DrugComb_to_ONeil source seed 42 budget 16 support draw 104 done (no query metrics printed)
DrugComb_to_ONeil source seed 42 budget 16 support draw 105 done (no query metrics printed)
DrugComb_to_ONeil source seed 42 budget 32 support draw 101 done (no query metrics printed)
DrugComb_to_ONeil source seed 42 budget 32 support draw 102 done (no query metrics printed)
DrugComb_to_ONeil source seed 42 budget 32 support draw 103 done (no query metrics printed)
DrugComb_to_ONeil source seed 42 budget 32 support draw 104 done (no query metrics printed)
DrugComb_to_ONeil source seed 42 budget 32 support draw 105 done (no query metrics printed)
DrugComb_to_ONeil source seed 42 budget 64 support draw 101 done (no query metri

,direction,budget,method,pcc_mean,pcc_run_sd,scc_mean,within_cell_pcc,ap_mean,auc_mean
10,DrugComb_to_ONeil,64,fewshot_pretrained_head,0.142653,0.038528,0.163301,0.169717,0.425703,0.592126
11,DrugComb_to_ONeil,64,fewshot_random_head,0.115993,0.072492,0.138496,0.145049,0.408101,0.565263
12,DrugComb_to_ONeil,64,residual_full_features_ridge,0.213357,0.035266,0.241826,0.236857,0.468207,0.626364
13,DrugComb_to_ONeil,64,source_only_sym,0.116392,0.014324,0.140283,0.133449,0.422126,0.586534
14,DrugComb_to_ONeil,64,target_only_fingerprint_ridge,0.244661,0.070322,0.277777,0.258392,0.490920,0.638053
30,ONeil_to_DrugComb,64,fewshot_pretrained_head,0.090231,0.022781,0.098127,0.041570,0.126799,0.638770
31,ONeil_to_DrugComb,64,fewshot_random_head,0.128792,0.045838,0.147484,0.051367,0.155565,0.684791
32,ONeil_to_DrugComb,64,residual_full_features_ridge,0.120045,0.041290,0.146169,0.071620,0.140893,0.703243
33,ONeil_to_DrugComb,64,source_only_sym,0.128686,0.001786,0.141314,0.091463,0.126892,0.706117
34,ONeil_to_DrugComb,64,target_only_fingerprint_ridge,0.178496,0.042478,0.155999,0.133116,0.149101,0.669509


,direction,budget,method_a,method_b,delta_pcc,ci_low,ci_high,query_pairs,warning
0,DrugComb_to_ONeil,64,fewshot_pretrained_head,source_only_sym,0.026261,0.012841,0.040400,283,Exploratory; shared query and few source seeds...
1,DrugComb_to_ONeil,64,fewshot_pretrained_head,fewshot_random_head,0.026661,-0.012080,0.062121,283,Exploratory; shared query and few source seeds...
2,DrugComb_to_ONeil,64,fewshot_pretrained_head,target_only_fingerprint_ridge,-0.102007,-0.160611,-0.046847,283,Exploratory; shared query and few source seeds...
3,DrugComb_to_ONeil,64,fewshot_pretrained_head,residual_full_features_ridge,-0.070704,-0.105776,-0.037387,283,Exploratory; shared query and few source seeds...
4,ONeil_to_DrugComb,64,fewshot_pretrained_head,source_only_sym,-0.038454,-0.062311,-0.013738,458,Exploratory; shared query and few source seeds...
5,ONeil_to_DrugComb,64,fewshot_pretrained_head,fewshot_random_head,-0.038561,-0.085226,0.011230,458,Exploratory; shared query and few source seeds...
6,ONeil_to_DrugComb,64,fewshot_pretrained_head,target_only_fingerprint_ridge,-0.088265,-0.143124,-0.033943,458,Exploratory; shared query and few source seeds...
7,ONeil_to_DrugComb,64,fewshot_pretrained_head,residual_full_features_ridge,-0.029814,-0.055225,-0.004414,458,Exploratory; shared query and few source seeds...


Support-CV selected configurations; epoch=0 means no neural update:


,method,mode,epoch,number_of_runs
0,fewshot_pretrained_head,last_layer,30,2
1,fewshot_pretrained_head,last_layer,60,1
2,fewshot_pretrained_head,last_layer,100,38
3,fewshot_pretrained_head,whole_head,10,12
4,fewshot_pretrained_head,whole_head,30,40
5,fewshot_pretrained_head,whole_head,60,18
6,fewshot_pretrained_head,whole_head,100,9
7,fewshot_random_head,last_layer,100,2
8,fewshot_random_head,whole_head,30,32
9,fewshot_random_head,whole_head,60,46


##Summary files
The summaries, the CV details and the audit files are packed into one small ZIP (`07_head_finetuning_summary.zip`). The feature caches, adapted head weights, job caches and compact NPZ predictions stay in the Drive folder `fewshot_head_finetune_v1/`. No large query-level CSV is written.

| File | Content |
|---|---|
| `all_metrics.csv` | all metrics for every run |
| `all_descriptive_summary.csv` | mean and SD (mixes support draws and source seeds; not independent tests) |
| `per_source_seed_summary.csv` | mean and support-draw SD for each source seed |
| `support_only_selections.csv`, `support_only_cv_trace.csv` | selected configurations and all inner-CV losses |
| `paired_primary_budget_comparisons.csv` | PCC differences at budget 64 with cluster-bootstrap intervals |
| `previous_control_reproduction_audit.csv`, `provenance_audit.csv` | reproduction and provenance checks |

The primary budget (64) was fixed in advance, and no budget or method is chosen after seeing the results. All results are reported, including those where fine-tuning does not help.

In [ ]:
# ===== Cell 6: Pack the summary files =====
# Only the small summary files are packed; features, head weights and predictions stay in Drive.
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
names=['analysis_plan.json','all_metrics.csv','all_descriptive_summary.csv',
       'per_source_seed_summary.csv','support_only_selections.csv','support_only_cv_trace.csv',
       'paired_primary_budget_comparisons.csv','previous_control_reproduction_audit.csv',
       'provenance_audit.csv','environment.txt']
missing=[name for name in names if not (OUTPUT/name).is_file()]
assert not missing, f'Run not complete; missing summaries: {missing}'
archive=Path('/content/07_head_finetuning_summary.zip')
with ZipFile(archive,'w',ZIP_DEFLATED) as z:
    for name in names:z.write(OUTPUT/name,name)
print(f'Summary ZIP: {archive.stat().st_size/1024**2:.2f} MB')
files.download(str(archive))

Summary ZIP: 0.08 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>